# Plaque-label HU measurements and separate vessel candidates

Corrected version of the uploaded notebook 13. Reports LAD, RCA, and LCX.
Model label 2 and unconfirmed vessel-adjacent HU candidates are measured separately;
candidates are never added to plaque totals. HU bins do not establish lesion composition.
Volumes use the actual image grid after mask geometry validation; derived-image spacing
is not replaced with spacing from a different series.

Research use only. Not clinically validated. Not for diagnosis.
Legacy masks without provenance are regenerated. If artery detection is ambiguous,
inspect the printed series inventory and explicitly set SERIES_OVERRIDES.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install the pinned library with runtime-compatible dependencies


In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = 'https://github.com/pazzani/OpenPlaque.git'
REPO_COMMIT = '26e35c03a7c556935773a2f5b611a7190caf28ec'
REPO_DIR = Path('/content') / f'OpenPlaque-{REPO_COMMIT}'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)
if subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip():
    # A new --no-checkout clone has deleted tracked files until its first checkout.
    if (REPO_DIR / 'src').exists():
        raise RuntimeError('Checkout contains local edits; use a fresh REPO_DIR.')
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert commit == REPO_COMMIT
# Load the stdlib-only installer before importing OpenPlaque or scientific packages.
import runpy
runpy.run_path(str(REPO_DIR / 'src/openplaque/colab_setup.py'))['install_dependencies'](REPO_DIR)
from importlib.metadata import version
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Pinned OpenPlaque commit:', commit)


## 3. Configure inputs and physical search radius


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = DRIVE_ROOT / 'Full_DICOM.zip'
MODEL_ZIP = DRIVE_ROOT / 'models' / 'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
OUTPUT_ROOT = DRIVE_ROOT / 'UCLA_Plaque_Type_Estimates_Audited'
VESSELS = ['LAD', 'RCA', 'LCX']
SERIES_OVERRIDES = {}  # Explicit selections only, e.g. {'LAD': 1043}; no silent fallback.
REUSE_SEGMENTATIONS = True
RUN_NNUNET_IF_MISSING = True
VESSEL_RADIUS_MM = 0.5  # Euclidean voxel-center distance, in physical millimeters.
HU_EDGES = (-30.0, 30.0, 130.0, 350.0)  # Lower bounds; last interval has no upper bound.
DATASET = 'Dataset001_CCTA_DHM'
FOLDS = (0, 1, 2, 3, 4)
print('Study:', STUDY_ZIP)


## 4. Verify the model and establish its identity


In [ ]:
import json
from openplaque.artery_estimates import extract_verified_zip, validate_model

model_root, model_zip_sha256 = extract_verified_zip(MODEL_ZIP, '/content/openplaque_model_cache')
# Google Drive archives sometimes wrap the nnU-Net directory in extra folders.
datasets = [p for p in model_root.rglob(DATASET) if p.is_dir()]
if len(datasets) != 1:
    raise ValueError(f'Expected exactly one {DATASET} folder in model ZIP; found {datasets}')
NNUNET_RESULTS = datasets[0].parent
model_files = validate_model(NNUNET_RESULTS, dataset=DATASET, folds=FOLDS)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
print('Verified model files:', list(model_files))


## 5. Load study and run or reuse segmentations


In [ ]:
import hashlib, tempfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import SimpleITK as sitk
from openplaque.artery_estimates import (
    ArteryEstimateStudy, select_artery_series, geometry, validate_geometry,
    load_cached_mask, save_cached_mask, file_sha256,
)
from openplaque.segmentation import SegmentationReport, export_for_nnunet, run_nnunet, load_segmentation

study = ArteryEstimateStudy(STUDY_ZIP)
study.summary()
series_map = select_artery_series(study, vessels=VESSELS, overrides=SERIES_OVERRIDES)
environment_snapshot = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
environment_sha256 = hashlib.sha256(environment_snapshot.encode()).hexdigest()
model_identity = {'zip_sha256': model_zip_sha256, 'files': model_files,
                  'dataset': DATASET, 'configuration': '3d_fullres', 'folds': list(FOLDS),
                  'nnunetv2': version('nnunetv2'), 'torch': version('torch'), 'library_commit': commit,
                  'python': sys.version, 'environment_sha256': environment_sha256}
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = OUTPUT_ROOT / study.study_sha256 / run_id
PLOT_DIR = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True)
MASK_DIR = OUTPUT_ROOT / 'mask_cache'
reports, provenance = [], []
for vessel in VESSELS:
    record = study.series_record(series_map[vessel])
    image, volume, files = study.load_series(series_map[vessel])
    identity = {'schema': 1, 'study_sha256': study.study_sha256,
                'series_uid': record['uid'], 'artery': vessel,
                'geometry': geometry(image), 'model': model_identity}
    cache_key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    saved_mask = MASK_DIR / cache_key / f'{vessel}.nii.gz'
    cached = None
    if REUSE_SEGMENTATIONS and saved_mask.exists():
        try:
            cached = load_cached_mask(saved_mask, image, identity)
        except (ValueError, OSError, RuntimeError) as exc:
            print(f'Rejected {vessel} mask cache: {exc}')
    if cached is not None:
        mask_image, mask = cached
        input_dir, output_dir = '', str(saved_mask.parent)
        source = 'verified_cache'
    else:
        if not RUN_NNUNET_IF_MISSING:
            raise FileNotFoundError(f'No verified mask for {vessel}; enable inference to regenerate it.')
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError('Enable a Colab GPU runtime for nnU-Net inference.')
        # Fresh per-inference directories prevent stale predictions from being accepted.
        with tempfile.TemporaryDirectory(prefix=f'openplaque-{vessel}-') as temp:
            input_dir, output_dir = str(Path(temp) / 'input'), str(Path(temp) / 'output')
            export_for_nnunet(image, input_dir, vessel)
            run_nnunet(input_dir, output_dir, dataset=DATASET, folds=FOLDS)
            mask_image, mask = load_segmentation(output_dir, vessel)
            validate_geometry(image, mask_image)
            save_cached_mask(saved_mask, image, mask_image, identity)
        input_dir, output_dir = '', str(saved_mask.parent)
        source = 'new_inference'
    reports.append(SegmentationReport(vessel, image, mask_image, volume, mask, input_dir, output_dir))
    provenance.append({'artery': vessel, 'series_number': series_map[vessel],
                       'series_uid': record['uid'], 'description': record['description'],
                       'geometry': geometry(image), 'mask_sha256': file_sha256(saved_mask),
                       'mask_source': source})
print('Series map:', series_map)


## 6. Measure model-labeled plaque and candidates separately

Each region is partitioned into the configured HU intervals, below-range voxels,
and nonfinite values. Model volume includes all label-2 voxels. Classified fractions
exclude below-range/nonfinite values and are undefined (blank) when no bins are populated.
TOTAL fractions are computed from volumes, allowing different artery voxel sizes.
No candidate volume is added to model plaque. Intermediate HU is not a mixed-lesion diagnosis.


In [ ]:
from openplaque.artery_estimates import summarize_artery, total_row, HU_KEYS

rows = [summarize_artery(r, radius_mm=VESSEL_RADIUS_MM, edges=HU_EDGES) for r in reports]
estimate_df = pd.DataFrame(rows + [total_row(rows)])
csv_path = OUTPUT_DIR / 'plaque_labels_and_candidates_by_artery.csv'
estimate_df.to_csv(csv_path, index=False)
for region, title in [('model', 'Model label 2'), ('candidate', 'Unconfirmed outer-shell candidates')]:
    print(title)
    columns = ['artery', f'{region}_region_mm3'] + [f'{region}_{k}_mm3' for k in HU_KEYS]
    columns += [f'{region}_below_range_mm3', f'{region}_nonfinite_mm3']
    display(estimate_df[columns])

freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUTPUT_DIR / 'environment.txt').write_text(freeze)
manifest = {'schema': 1, 'created_utc': run_id, 'library_commit': commit,
            'study_sha256': study.study_sha256, 'model': model_identity,
            'series': provenance, 'series_overrides': SERIES_OVERRIDES,
            'radius_mm': VESSEL_RADIUS_MM, 'hu_edges': list(HU_EDGES),
            'python': sys.version, 'volume_basis': 'validated image grid spacing',
            'candidate_policy': 'separate; never added to model plaque',
            'fraction_denominator': 'classified volume within each region',
            'total_scope': 'sum of selected artery grids; anatomical non-overlap requires review'}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))


## 7. Plot and export the auditable report


In [ ]:
import base64, html
import matplotlib.pyplot as plt

labels = [f'{HU_EDGES[i]:g} to <{HU_EDGES[i+1]:g} HU' if i < 3 else f'>={HU_EDGES[i]:g} HU'
          for i in range(4)]
plot_df = estimate_df[estimate_df.artery != 'TOTAL'].set_index('artery')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, region, title in zip(axes, ['model', 'candidate'],
                            ['Model-labeled plaque: HU bins', 'Unconfirmed outer-shell candidates: HU bins']):
    columns = [f'{region}_{k}_mm3' for k in HU_KEYS]
    columns += [f'{region}_below_range_mm3', f'{region}_nonfinite_mm3']
    plot_df[columns].plot.bar(stacked=True, ax=ax, edgecolor='black')
    ax.set_title(title)
    ax.set_ylabel('Image-grid volume (mm3)')
    ax.legend(labels + [f'<{HU_EDGES[0]:g} HU', 'Nonfinite'], fontsize=8)
fig.tight_layout()
plot_path = PLOT_DIR / 'plaque_labels_and_candidates.png'
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)

sections = []
for region, title in [('model', 'Model label 2'), ('candidate', 'Unconfirmed candidates')]:
    cols = ['artery'] + [c for c in estimate_df.columns if c.startswith(region + '_') and not c.endswith('_voxels')]
    sections.append(f'<h2>{title}</h2>' + estimate_df[cols].to_html(index=False, na_rep='N/A', float_format=lambda x: f'{x:.4g}'))
image_data = base64.b64encode(plot_path.read_bytes()).decode('ascii')
html_path = OUTPUT_DIR / 'plaque_labels_and_candidates_by_artery.html'
report_html = f"""<!doctype html><html><head><meta charset="utf-8"><title>Artery HU measurements</title>
<style>body{{font-family:sans-serif;margin:28px}}table{{border-collapse:collapse;font-size:12px}}td,th{{border:1px solid #ccc;padding:6px}}img{{max-width:100%}}</style></head><body>
<h1>Model plaque labels and separate HU candidates</h1>
<p>Research use only. Not clinically validated. Candidate voxels are not confirmed plaque and are never added to model totals.
HU bins describe intensity, not mixed-lesion composition. Volumes use the validated image grid; physical validity of derived reformats and anatomical non-overlap require review.</p>
<p>Outer-shell radius: {VESSEL_RADIUS_MM:g} mm (Euclidean voxel-center distance). HU bins: {html.escape('; '.join(labels))}.
Below-range and nonfinite voxels are shown separately. Fractions use classified region volume; N/A means no classified volume.</p>
{''.join(sections)}<img alt="Separate model and candidate volume charts" src="data:image/png;base64,{image_data}">
<p><a href="{csv_path.name}">Full CSV</a> | <a href="manifest.json">Run provenance</a> | <a href="environment.txt">Environment</a></p>
<h2>Run provenance</h2><pre>{html.escape(json.dumps(manifest, indent=2))}</pre></body></html>"""
html_path.write_text(report_html, encoding='utf-8')
print('CSV:', csv_path)
print('HTML:', html_path)
print('Plot:', plot_path)
